In [ ]:
import numpy as np
import math
import sys

SIZE = 5
DEGREE = 2
GOAL = np.array([(0,0)])
EPISODES = 10000
STEP_SIZE = 5*(10**-4)
DISCOUNT = .9
SHOW_EPISODE = 1


def main():
    epsilon = 1
    average = 0
    # [x0y0, x0y1, x0y2, x1y0, x1y1, x1y2, x2y0, x2y1, x2y2]
    weights = np.full(((DEGREE+1),(DEGREE+1)), 0.0)

    for current in range(EPISODES):
        pos = (SIZE-1,SIZE-1)
        count = 0

        while(find_reward(pos)!=1):
            #if(current==5 and count==1):
                #sys.exit()

            a = pick_action(pos,epsilon,weights)
            action = (round(math.cos(a)),round(math.sin(a)))
            new_pos = find_new(action,pos)

            #if(current>=1 or find_reward(new_pos)==1):
                #print(pos)

            features = np.empty((DEGREE+1,DEGREE+1))
            next_features = np.empty((DEGREE+1,DEGREE+1))
            reward = find_reward(new_pos)
            for x in range(DEGREE+1):
                for y in range(DEGREE+1):
                    features[x][y] = (pos[0]**x)*(pos[1]**y)
                    next_features[x][y] = (new_pos[0]**x)*(new_pos[1]**y)

            old_value = np.sum(np.multiply(weights,features))
            new_value = np.sum(np.multiply(weights,next_features))

            weights = np.subtract(weights,STEP_SIZE*(reward+DISCOUNT*new_value-old_value)*(np.subtract(DISCOUNT*next_features,features)))
            #residual = old_value-(reward+DISCOUNT*new_value)
            #weights = np.subtract(weights,STEP_SIZE*(np.subtract(2*residual*features,2*DISCOUNT*residual*next_features)))

            pos = new_pos
            count += 1

            #if(current>=1 and count%1==0):
                #print_weights(10,weights)

        if((current+1)%SHOW_EPISODE==0):
            print_weights(4,weights)
            print(average)
            average=0
        else:
            average = (average*(current%SHOW_EPISODE)+count)/((current+1)%SHOW_EPISODE)
        epsilon = epsilon - epsilon/(EPISODES//2-1)

    print_episode(weights)


def pick_action(pos,epsilon,weights):
    if(np.random.random()<epsilon):
        return(np.random.choice([0,np.pi/2,np.pi,3*np.pi/2]))
    else:
        return((np.pi/2)*arg_max(pos,weights))

def arg_max(pos,weights):
    values = np.empty(4)
    for i in range(4):
        new_pos = (pos[0]+round(math.cos(np.pi/2*i)),pos[1]+round(math.sin(np.pi/2*i)))
        if((new_pos[0]>=0) and (new_pos[0]<=SIZE-1) and (new_pos[1]>=0) and (new_pos[1]<=SIZE-1)):
            values[i] = find_value(new_pos,weights)
        else:
            values[i] = -math.inf
    return(np.argmax(values))

def find_value(pos,weights):
    temp = 0
    for x in range(DEGREE+1):
        for y in range(DEGREE+1):
            temp += (weights[x][y])*((pos[0])**x)*((pos[1])**y)
    return(temp)

def find_new(action,pos):
    temp1 = action[0]+pos[0]
    if(temp1>=SIZE):
        temp1 = SIZE-1
    elif(temp1<0):
        temp1 = 0
    temp2 = action[1]+pos[1]
    if(temp2>=SIZE):
        temp2 = SIZE-1
    elif(temp2<0):
        temp2 = 0
    return(temp1,temp2)

def find_reward(pos):
    check = False
    for g in GOAL:
        if(g[0]==pos[0] and g[1]==pos[1]):
            check = True
    if(check):
        return 1
    else:
        return 0

def print_weights(d_places,weights):
    print("[", end="")
    for x in range(DEGREE+1):
        for y in range(DEGREE+1):
            if(not((x==DEGREE)and(y==DEGREE))):
                print(round(weights[x][y],d_places), end=", ")
            else:
                print(round(weights[x][y],d_places), end="")
    print("]")

def print_episode(weights):
    for y in range(SIZE):
        line = ""
        for x in range(SIZE):
            a = arg_max((x,y),weights)
            if(a==0):
                line += "> "
            elif(a==1):
                line += "V "
            elif(a==2):
                line += "< "
            else:
                line += "^ "
        print(line+"\n")


main()